In [ ]:
!pip install umap hdbscan bertopic jiwer

In [ ]:
import os
import sys
import shutil
# Detect if running in Google Colab

# Set the environment variable for your GitHub token
#os.environ["GITHUB_TOKEN"] =

# This cell is for loading data. If your prefer to do this manually, you will need to set base_dir and data_dir separately

IN_COLAB = 'google.colab' in sys.modules

# Check if running in Google Colab
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Set the base directory on Google Drive (no extra folder will be added)
    base_dir = "/content/drive/MyDrive/Bertopic"
    token = os.getenv("GITHUB_TOKEN")
    #if os.path.exists(base_dir):
     #   shutil.rmtree(base_dir)

    #!git clone https://{token}@github.com/UnbrokenCocoon/OCR-evaluation.git "{base_dir}"

else:
    # Set the base directory locally (set this to your local project folder)
    base_dir = "path/to/your/local/project/folder"

    #!git clone https://{token}@github.com/UnbrokenCocoon/OCR-evaluation.git "{base_dir}"

    # Clone the repository locally


# Set the data directory (this assumes you have a 'Data' folder inside the repository)
data_dir = os.path.join(base_dir, "Data")
output_dir = os.path.join(base_dir, "output")

# Now data_dir points to the cloned Data folder
print(f"Data folder is located at: {data_dir}")


In [ ]:
# Load in the sentences
import pickle
import pandas as pd
with open(os.path.join(data_dir, r'bs_emb.pkl'), "rb") as f:
    embeddings = pickle.load(f):
with open(os.path.join(data_dir, r'bs_sen.pkl'), "rb") as f:
    all_sentences = pickle.load(f)
df_topic_freq = pd.read_csv(os.path.join(data_dir,'bs topic freq.csv'))

In [ ]:
# Compute the term freqency dictionary
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

# Step 1: Create and fit the vectorizer
vectorizer = CountVectorizer(
    stop_words="english",
    min_df=2,
    ngram_range=(1, 2),
    token_pattern=r"(?u)\b\w{3,}\b"
)

X = vectorizer.fit_transform(all_sentences)

# Step 2: Get token names and their total frequencies
terms = vectorizer.get_feature_names_out()
frequencies = np.asarray(X.sum(axis=0)).flatten()

# Step 3: Create the dictionary
term_freq_dict = dict(zip(terms, frequencies))
total_term_freq = sum(frequencies)
# Example: print the top 10 most frequent terms
top_10 = sorted(term_freq_dict.items(), key=lambda x: x[1], reverse=True)[:10]
print("🔝 Top 10 terms by frequency:")
for term, freq in top_10:
    print(f"{term:<20} : {freq}")


In [ ]:
# Set up the run
import random
import copy
import numpy as np
import pandas as pd
import ast
import re
from jiwer import wer
from bertopic import BERTopic
from umap.umap_ import UMAP
from hdbscan import HDBSCAN
import gc
# --- Gini function ---
def gini(array):
    array = np.sort(np.array(array))
    n = len(array)
    if np.sum(array) == 0:
        return 0.0
    index = np.arange(1, n + 1)
    return ((np.sum((2 * index - n - 1) * array)) / (n * np.sum(array)))

# --- Assumes these are preloaded ---
# all_sentences, embeddings, term_freq_dict, total_term_freq, df_topic_freq

# --- Initialise ---
best_error_size = 1_000_000
best_topic20_size = 0
best_model = None
best_top_df = None

error_sizes = []
topic20_sizes = []
gini_scores = []
ngram_values = []


In [ ]:
# --- Loop ---
for i in range(60):
    print(f"\n🔁 Iteration {i+1}")

    min_cluster_size = random.choice(range(5,80,3))
    min_topic_size = random.choice(range(5,80,3))

    hdbscan_model = HDBSCAN(min_cluster_size=min_cluster_size, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
    umap_model = UMAP(n_neighbors=random.choice([5,10,15]), n_components=5, min_dist=0.0, metric='cosine', random_state=42)
    vectorizer_model = CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2), token_pattern=r"(?u)\b\w{3,}\b")

    topic_model = BERTopic(
        embedding_model=None,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        min_topic_size=min_topic_size,
        nr_topics=50,
        top_n_words=10,
        verbose=False
    )

    topics, probs = topic_model.fit_transform(all_sentences, embeddings)
    topic_info = topic_model.get_topic_info()
    error_size = topic_info['Count'].iloc[0]

    if error_size in error_sizes:
        continue

    if len(topic_info) > 19:
        topic_info_filtered = topic_info[topic_info["Topic"] != -1]
        top_df = topic_info_filtered.nlargest(20, "Count").copy()
        topic20_size = topic_info['Count'].iloc[19]
    else:
        top_df = pd.DataFrame()
        topic20_size = 0

    # --- Gini score ---
    # --- Keyword frequency score ---
    ngram_value = 0
    if not top_df.empty:
        topic_counts = top_df["Count"].values
        gini_score = gini(topic_counts)

        for topic_id in top_df['Topic'][:20]:
            for word, _ in topic_model.get_topic(topic_id):
                if word in term_freq_dict:
                    ngram_value += term_freq_dict[word]
    else:
        gini_score = 0


    # --- Save metrics ---
    error_sizes.append(error_size)
    topic20_sizes.append(topic20_size)
    ngram_values.append(round(ngram_value / total_term_freq, 2))
    gini_scores.append(gini_score)


    print(f"📊 error_size: {error_size}, topic_20_size: {topic20_size}, gini: {gini_score:.3f},  keyword freq score: {ngram_value}")
    mean_error_size = np.mean(error_sizes)
    if error_size <= mean_error_size * 1.1 and topic20_size >= best_topic20_size * 0.95:
        print("📌 Model within fuzzy threshold — saving as best candidate.")
        best_ngram = round(ngram_value / total_term_freq, 2)
        best_gini = gini_score
        best_error_size = error_size
        best_topic20_size = topic20_size
        best_model = copy.deepcopy(topic_model)
        best_top_df = top_df.copy()
    gc.collect()
# --- Summary ---
print("\n✅ Done. Best model and top_df are stored in `best_model` and `best_top_df`.")
print(f"📈 Average error size: {np.mean(error_sizes):.2f}")
print(f"📈 Average topic 20 size: {np.mean(topic20_sizes):.2f}")
print(f"📉 Average Gini: {np.mean(gini_scores):.3f}")
print(f"📈 Average keyword freq score: {np.mean(ngram_values):.2f}")
print(f"🔹 Best error size: {best_error_size}")
print(f"🔹 Best topic 20 size: {best_topic20_size}")

# --- Final metrics DataFrame ---
metrics_df = pd.DataFrame({
    "Iteration": np.arange(1, len(error_sizes) + 1),
    "Error_Size": error_sizes,
    "Topic_20_Size": topic20_sizes,
    "Gini_Score": gini_scores,
    "Keyword_Freq_Score": ngram_values
})


In [ ]:
print(len(metrics_df))
# optionally save metrics_df

In [ ]:
# Run for calculating PUVs

error_sizes = []
puv_scores = []
params_used = []
import spacy
nlp = spacy.load("en_core_web_sm", disable = ['parser','ner'])

for i in range(30):
    print(f"\n🔁 Iteration {i+1}")

    # 🔧 Randomise parameters
    min_cluster_size = random.choice([25, 35, 40, 45, 50])
    min_topic_size = random.choice([10, 15, 20, 25, 30, 35, 40, 45, 50])

    # ⚙️ Configure models
    hdbscan_model = HDBSCAN(
        min_cluster_size=min_cluster_size,
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True
    )
    umap_model = UMAP(
        n_neighbors=5,
        n_components=5,
        min_dist=0.0,
        metric='cosine',
        random_state=42
    )
    vectorizer_model = CountVectorizer(
        stop_words="english",
        min_df=2,
        ngram_range=(1, 2),
        token_pattern=r"(?u)\b\w{3,}\b"
    )

    # 🧠 Run BERTopic
    topic_model = BERTopic(
        embedding_model=None,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        min_topic_size=min_topic_size,
        nr_topics=50,
        top_n_words=10,
        verbose=False
    )

    topics, probs = topic_model.fit_transform(all_sentences, embeddings)
    topic_info = topic_model.get_topic_info()
    topics_dict = topic_model.get_topics()
    error_size = topic_info['Count'].iloc[0]  # topic -1

    # 🔍 Extract top terms for topics 0–19
    raw_ngrams = []
    for topic_id, terms in topics_dict.items():
        if topic_id < 0 or topic_id > 19:
            continue
        for term, _ in terms:
            raw_ngrams.append(term)

    # ✨ Lemmatise all terms in one batch
    doc = nlp(" ".join(raw_ngrams))
    top_ngrams = [token.lemma_ for token in doc if not token.is_space]

    # 📊 Calculate PUV
    if top_ngrams:
        counts = Counter(top_ngrams).most_common()
        copied_ngrams = [ngram for ngram, freq in counts if freq > 1]
        puv = len(copied_ngrams) / len(top_ngrams)
    else:
        puv = 0.0

    print(f"📊 error_size: {error_size}, puv: {puv:.3f}")

    # 📥 Store results
    error_sizes.append(error_size)
    puv_scores.append(puv)
    params_used.append({
        "min_cluster_size": min_cluster_size,
        "min_topic_size": min_topic_size
    })

    gc.collect()

# ✅ Combine into a DataFrame
results_df = pd.DataFrame({
    "iteration": list(range(1, 31)),
    "error_size": error_sizes,
    "puv_score": puv_scores,
    "min_cluster_size": [p["min_cluster_size"] for p in params_used],
    "min_topic_size": [p["min_topic_size"] for p in params_used]
})

# Optional: Display or export
print("\n📈 Summary of Results:")
print(results_df.describe())
# results_df.to_csv("bertopic_eval_results.csv", index=False)


In [ ]:
import pandas as pd

# Assume these variables are already defined:
# best_gini, best_error_size, best_topic20_size, PUV, ngram_value, all_sentences

ngram_value =0
top_df = best_top_df.head(20).copy()
for topic_id in top_df['Topic'][:20]:
            for word, _ in topic_model.get_topic(topic_id):
                if word in term_freq_dict:
                    ngram_value += term_freq_dict[word]
ngram_value= round(ngram_value / total_term_freq, 2)

df_metrics = pd.DataFrame([{
    "Gini Score": best_gini,
    "Appearance Percentage": (1 - best_error_size / len(all_sentences)) * 100,
    "Topic 20 Size": best_topic20_size,
    "PUV": PUV,
    "Ngram Value": ngram_value
}])

# Round all numeric values to 2 decimal places
df_metrics = df_metrics.round(2)

# Output as markdown table
print(df_metrics.to_markdown())


In [ ]:
# Look at the topics and counts visually
# First create custom_list of variable names
top_df = best_top_df.head(15).copy()
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.barh(custom_list, top_df["Count"]/len(all_sentences)*100, color="darkgreen")
plt.xlabel("Topic Frequency in sentences (%)")
plt.ylabel("Topic Name")
plt.title("Top Clusters")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Visualize the top N topics (e.g., top 5 topics)
# First set custom topic labels
fig = best_model.visualize_barchart(top_n_topics=15, width =300, autoscale = True, custom_labels=True)
# Customize the chart (e.g., add a title)
fig.update_layout(
    title="Top 15 Topics by Word Frequency"
)
# Show the chart
fig.write_html("topic_barchart.html")
fig.show()

In [ ]:
# Examine the colo[u]rs set to make sure you can see each one clearly
# This is an important step to make sure that the bertopic is readable
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
colors = [
    "#2F4F4F",  # Dark Slate Grey
    "#F5D0C5",  # Lighter, warmer peach
    "#800000",  # Dark Red
    "#DAA520",  # Goldenrod
    "#00008B",  # Dark Blue
    "#AEEEEE",  # Lighter Cyan (increase saturation)
    "#7FFF00",  # Chartreuse (bright green)
    "#DC143C",  # Crimson (dark red)
    "#FF8C00",  # Dark Orange
    "#AADC22",  # Lighter Green (increase saturation)
    "#4EC0AD",  # Slightly brighter teal
    "#EE44EE",  # Brighter purple
    "#B23DEE",  # Slightly richer purple
    "#335499",  # Bright Blue
    "#A3FFFF"   # Very light cyan (increase saturation)
]
# Create color patches
patches = [mpatches.Patch(color=color, label=color) for color in colors]  # Use topic_colors

# Display the patches
plt.legend(handles=patches, loc='center')
plt.axis('off')  # Turn off axes
plt.show()

In [ ]:
import itertools
import pandas as pd
topic_model = best_model.copy()

umap_model = UMAP(
    n_neighbors=10,
    n_components=2,
    min_dist=0.2,  # Try 0.2–0.3 for clearer spread
    metric='cosine',
    random_state=42
)
reduced_embeddings_2d = umap_model.fit_transform(embeddings)


# Step 3: Get top 20 topics by frequency

colors = itertools.cycle(colors)

# Define colors for the visualization to iterate over
top_topics = set(top_df["Topic"].astype(str))
color_key = {topic: next(colors) for topic in top_topics}

doc_topics = [str(t) for t in best_model.topics_]

# Full UMAP plot dataframe, filtered to Top 20 topics
df = pd.DataFrame({
    "x": reduced_embeddings_2d[:, 0],
    "y": reduced_embeddings_2d[:, 1],
    "Topic": doc_topics,
    "Length": [len(doc) for doc in all_sentences]
})

df = df[df["Topic"].isin(top_topics)]
df = df[(df.y > -10) & (df.y < 10) & (df.x > -10) & (df.x < 10)]
df["Topic"] = df["Topic"].astype("category")

mean_df = df.groupby("Topic").mean(numeric_only=True).reset_index()
mean_df["Topic"] = mean_df["Topic"].astype(int)
mean_df = mean_df.sort_values("Topic")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(10, 8))

# Plot each topic cluster with correct colour
for topic_id in df["Topic"].cat.categories:
    topic_df = df[df["Topic"] == topic_id]
    label = topic_id_to_label[int(topic_id)]
    colour = color_key[str(topic_id)]

    ax.scatter(
        topic_df["x"],
        topic_df["y"],
        label=label,
        color=colour,
        alpha=0.6,
        s=10
    )

# Build legend manually to maintain label-colour mapping
handles = [
    mpatches.Patch(color=color_key[str(topic_id)], label=topic_id_to_label[int(topic_id)])
    for topic_id in df["Topic"].cat.categories
]

ax.legend(
    handles=handles,
    loc='best',
    title="Topics",
    fontsize=10,
    title_fontsize=11
)

ax.set_title("UMAP Projection of Topics", fontsize=14)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")

plt.tight_layout()
plt.show()
